## Structure of this notebook

There are 4 phases to this notebook. Each phase handles a different task in the engineering of a trade dependency variable.

Each phase, a CSV will be saved to **'data/processed/trade_dependency_engineering/p[]_trade_engineering.csv'**, acting as a checkpoint.

This makes it easier to track changes between each phase.

<br>

| Phase | Actions | Columns | CSV filename |
|----------|---------|---------------------|--------|
| 1 | Summing bilateral trade flows to get one value per undirected pairxyear | year, total_value_usd, country_a, country_b, a_iso3, b_iso3 | p1_trade_engineering.csv |
| 2 | Add GDP per capita and calculate GDP-based dependency | + a_gdp, b_gdp, dependency_a_pct, dependency_b_pct | p2_trade_engineering.csv |
| 3 | Add income classification | + a_ic, b_ic | p3_trade_engineering.csv |
| 4 | Calculate trade share of trade with all partners | total_value_usd->total_trade_relation_value + a_total_trade_all_partners, b_total_trade_all_partners, a_trade_share, b_trade share | p4_trade_engineering.csv |

### Phase 1: Calculating total trade value for each year+country pairing

In [129]:
import pandas as pd

df = pd.read_csv('../data/processed/v3_imf_trade.csv') 

Step 1: Check the distribution of 'export' and 'import' in the 'export_or_import' column in v3_imf_trade.csv (trade in goods dataset)

In [130]:
# Check the count of export vs import rows
print("=" * 50)
print("TRADE DIRECTION COUNTS")
print("=" * 50)

direction_counts = df['export_or_import'].value_counts()

print(f"\nExports: {direction_counts.get('export', 0):,} rows")
print(f"Imports: {direction_counts.get('import', 0):,} rows")
print(f"Total: {len(df):,} rows")

# Calculate percentages
total = len(df)
if 'export' in direction_counts:
    export_pct = (direction_counts['export'] / total) * 100
    print(f"\nExports: {export_pct:.1f}% of total")
if 'import' in direction_counts:
    import_pct = (direction_counts['import'] / total) * 100
    print(f"Imports: {import_pct:.1f}% of total")

TRADE DIRECTION COUNTS

Exports: 718,147 rows
Imports: 53,384 rows
Total: 771,531 rows

Exports: 93.1% of total
Imports: 6.9% of total


Step 2: Conduct a flow analysis

Warning: Don't run this code block!!! It takes ~10 minutes.

- The importance is in the output (that is hopefully there for you to read)

- If the output is missing, navigate to 'imf_trade_v3_flow_analysis.txt' in /outputs

- You can run the blocks below it perfectly fine

In [131]:
import pandas as pd

df = pd.read_csv('../data/processed/v3_imf_trade.csv')

# Create a flag for each row
df['direction'] = df['country_a'] + '→' + df['country_b']
df['reverse_direction'] = df['country_b'] + '→' + df['country_a']

# For each undirected pair and year, count what exists
def classify_pair(group):
    """Classify what trade flows exist for a pair in a given year"""
    
    # Check the four possible flows
    has_a_to_b_export = len(group[(group['country_a'] == group.iloc[0]['country_a']) & 
                                   (group['country_b'] == group.iloc[0]['country_b']) & 
                                   (group['export_or_import'] == 'export')]) > 0
    
    has_a_to_b_import = len(group[(group['country_a'] == group.iloc[0]['country_a']) & 
                                   (group['country_b'] == group.iloc[0]['country_b']) & 
                                   (group['export_or_import'] == 'import')]) > 0
    
    has_b_to_a_export = len(group[(group['country_a'] == group.iloc[0]['country_b']) & 
                                   (group['country_b'] == group.iloc[0]['country_a']) & 
                                   (group['export_or_import'] == 'export')]) > 0
    
    has_b_to_a_import = len(group[(group['country_a'] == group.iloc[0]['country_b']) & 
                                   (group['country_b'] == group.iloc[0]['country_a']) & 
                                   (group['export_or_import'] == 'import')]) > 0
    
    # Count how many exist
    flows = [has_a_to_b_export, has_a_to_b_import, has_b_to_a_export, has_b_to_a_import]
    count = sum(flows)
    
    # Classify the pattern
    if count == 4:
        pattern = "all 4 flows (both directions, both types)"
    elif count == 3:
        pattern = "3 flows"
    elif count == 2:
        if has_a_to_b_export and has_a_to_b_import:
            pattern = "2 flows: both types in A→B only"
        elif has_b_to_a_export and has_b_to_a_import:
            pattern = "2 flows: both types in B→A only"
        elif has_a_to_b_export and has_b_to_a_export:
            pattern = "2 flows: exports only (both directions)"
        elif has_a_to_b_import and has_b_to_a_import:
            pattern = "2 flows: imports only (both directions)"
        elif has_a_to_b_export and has_b_to_a_import:
            pattern = "2 flows: A→B export + B→A import"
        else:
            pattern = "2 flows: other combination"
    elif count == 1:
        if has_a_to_b_export:
            pattern = "1 flow: A→B export only"
        elif has_a_to_b_import:
            pattern = "1 flow: A→B import only"
        elif has_b_to_a_export:
            pattern = "1 flow: B→A export only"
        else:
            pattern = "1 flow: B→A import only"
    else:
        pattern = "0 flows (should not exist in data)"
    
    return pd.Series({
        'flow_count': count,
        'pattern': pattern
    })

# Create undirected pair identifier
df['pair_undirected'] = df.apply(lambda x: tuple(sorted([x['country_a'], x['country_b']])), axis=1)

# Group by year and undirected pair
classification = df.groupby(['year', 'pair_undirected']).apply(classify_pair).reset_index()

print("=" * 60)
print("TRADE FLOW PATTERN ANALYSIS")
print("=" * 60)

# Count each pattern
pattern_counts = classification['pattern'].value_counts()
print("\nPattern distribution:")
for pattern, count in pattern_counts.items():
    pct = (count / len(classification)) * 100
    print(f"  {pattern}: {count:,} pairs ({pct:.1f}%)")

# Count by number of flows
flow_count_distribution = classification['flow_count'].value_counts().sort_index()
print("\n\nBy number of flows:")
for flow_count, count in flow_count_distribution.items():
    pct = (count / len(classification)) * 100
    print(f"  {flow_count} flows: {count:,} pairs ({pct:.1f}%)")

# Summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total unique year-country pairs: {len(classification):,}")
print(f"Pairs with both directions (any type): {len(classification[classification['flow_count'] >= 2]):,}")
print(f"Pairs with complete data (all 4 flows): {len(classification[classification['flow_count'] == 4]):,}")
print(f"Pairs with only exports (both directions): {len(classification[classification['pattern'] == '2 flows: exports only (both directions)']):,}")
print(f"Pairs with only imports (both directions): {len(classification[classification['pattern'] == '2 flows: imports only (both directions)']):,}")

C:\Users\vanes\AppData\Local\Temp\ipykernel_10572\3833215918.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  classification = df.groupby(['year', 'pair_undirected']).apply(classify_pair).reset_index()


TRADE FLOW PATTERN ANALYSIS

Pattern distribution:
  2 flows: exports only (both directions): 239,747 pairs (54.1%)
  1 flow: A→B export only: 151,664 pairs (34.2%)
  3 flows: 36,830 pairs (8.3%)
  2 flows: both types in A→B only: 5,909 pairs (1.3%)
  2 flows: A→B export + B→A import: 2,839 pairs (0.6%)
  1 flow: A→B import only: 2,787 pairs (0.6%)
  2 flows: other combination: 2,106 pairs (0.5%)
  all 4 flows (both directions, both types): 1,340 pairs (0.3%)
  2 flows: imports only (both directions): 14 pairs (0.0%)


By number of flows:
  1 flows: 154,451 pairs (34.8%)
  2 flows: 250,615 pairs (56.5%)
  3 flows: 36,830 pairs (8.3%)
  4 flows: 1,340 pairs (0.3%)

SUMMARY
Total unique year-country pairs: 443,236
Pairs with both directions (any type): 288,785
Pairs with complete data (all 4 flows): 1,340
Pairs with only exports (both directions): 239,747
Pairs with only imports (both directions): 14


From the analysis, there are 288k cases where both trade flows are captured for that year and country pairing. In those cases both trade flows exist but can look differently in the data.

Let's handle scenario 1, the cleanest cases

#### Scenario 1: Exports only (both flows captured), ~54% of cases

Step 1.1 Create a dataframe that only has export records

In [133]:
df = pd.read_csv('../data/processed/v3_imf_trade.csv')

exports = df[df['export_or_import'] == 'export'].copy()

Step 1.3 Check for duplicate export records

In [134]:
duplicates = exports.groupby(['year', 'country_a', 'country_b']).size()
print(f"Ordered pairs with duplicate exports: {(duplicates > 1).sum()}")

Ordered pairs with duplicate exports: 0


Step 1.2 Create a new column that holds a tuple where both country names are sorted alphabetically. 

This essentially makes us group China->US and US->China together. It gives us a column to see the undirected pair.

In [136]:
exports['pair_undirected'] = exports.apply(lambda x: tuple(sorted([x['country_a'], x['country_b']])), axis=1)

Step 1.3 Count export records per undirected pair (that we just made the identifier for) and year

In [137]:
export_counts = exports.groupby(['year', 'pair_undirected']).size().reset_index(name='export_count')

Step 1.4 Filter out the pairs with exactly 2 export records per year

clean_pairs now is a list of (year, pair_undirected), that have exactly 2 export records

In [138]:
clean_pairs = export_counts[export_counts['export_count'] == 2][['year', 'pair_undirected']]

Step 1.5 Merge 'clean_pairs' with 'exports' dataframe to get 'clean_exports', containing (year, pair_undirected, usd_value, ...)

In [139]:
clean_exports = exports.merge(clean_pairs, on=['year', 'pair_undirected'])

clean_exports.head(3)

,country_a,export_or_import,country_b,year,usd_value,a_iso3,b_iso3,pair_undirected
0,"Uzbekistan, Republic of",export,American Samoa,2021.0,0.007679,UZB,ASM,"(American Samoa, Uzbekistan, Republic of)"
1,"Uzbekistan, Republic of",export,American Samoa,2023.0,0.000048,UZB,ASM,"(American Samoa, Uzbekistan, Republic of)"
2,"Iran, Islamic Republic of",export,Togo,2000.0,0.094464,IRN,TGO,"(Iran, Islamic Republic of, Togo)"


In [140]:
# check = clean_exports[(clean_exports['pair_undirected'] == ('American Samoa', 'Uzbekistan, Republic of'))].copy()

# check.head(40)

Step 1.6: Sum the two export values for each pairing+year

We can do that because we confirmed that there are only 2 export records (1 for each direction). Both trade flows will be captured

In [141]:
scen1_summed_trade = clean_exports.groupby(['year', 'pair_undirected'], as_index=False).agg(
    total_value_usd = ('usd_value', 'sum')
)

Step 1.7: Put the countries from 'pair_undirected' back into variables country_a and country_b

In [142]:
scen1_summed_trade[['country_a', 'country_b']] = pd.DataFrame(scen1_summed_trade['pair_undirected'].tolist(), index=scen1_summed_trade.index)

scen1_summed_trade.head(3)

,year,pair_undirected,total_value_usd,country_a,country_b
0,1992.0,"(Afghanistan, Islamic Republic of, Australia)",0.223057,"Afghanistan, Islamic Republic of",Australia
1,1992.0,"(Afghanistan, Islamic Republic of, Austria)",0.983654,"Afghanistan, Islamic Republic of",Austria
2,1992.0,"(Afghanistan, Islamic Republic of, Bangladesh)",0.450487,"Afghanistan, Islamic Republic of",Bangladesh


Check to compare

This is how the two rows in the unsummed dataframe look like:

In [143]:
check = clean_exports[(clean_exports['pair_undirected'] == ('American Samoa', 'Uzbekistan, Republic of')) & (clean_exports['year'] == 2021)].copy()

check.head(40)

,country_a,export_or_import,country_b,year,usd_value,a_iso3,b_iso3,pair_undirected
0,"Uzbekistan, Republic of",export,American Samoa,2021.0,0.007679,UZB,ASM,"(American Samoa, Uzbekistan, Republic of)"
457,American Samoa,export,"Uzbekistan, Republic of",2021.0,0.000080,ASM,UZB,"(American Samoa, Uzbekistan, Republic of)"


This is how those two rows look after summing them into one:

In [144]:
check = scen1_summed_trade[(scen1_summed_trade['pair_undirected'] == ('American Samoa', 'Uzbekistan, Republic of')) & (scen1_summed_trade['year'] == 2021)].copy()

check.head(40)

,year,pair_undirected,total_value_usd,country_a,country_b
237659,2021.0,"(American Samoa, Uzbekistan, Republic of)",0.007759,American Samoa,"Uzbekistan, Republic of"


Step 1.8: Drop pair_undirected

In [145]:
scen1_summed_trade = scen1_summed_trade.drop('pair_undirected', axis=1)

#### Scenario 2: Imports only (both flows captured), only 14 pairs

Step 2.1 Create a dataframe that only has import records

In [146]:
df = pd.read_csv('../data/processed/v3_imf_trade.csv')

imports = df[df['export_or_import'] == 'import'].copy()

Step 1.3 Check for duplicate import records

In [147]:
duplicates = imports.groupby(['year', 'country_a', 'country_b']).size()
print(f"Ordered pairs with duplicate imports: {(duplicates > 1).sum()}")

Ordered pairs with duplicate imports: 0


Step 1.2 Create a new column that holds a tuple where both country names are sorted alphabetically. 

This essentially makes us group China->US and US->China together. It gives us a column to see the undirected pair.

In [148]:
imports['pair_undirected'] = imports.apply(lambda x: tuple(sorted([x['country_a'], x['country_b']])), axis=1)

Step 1.3 Count import records per undirected pair (that we just made the identifier for) and year

In [149]:
import_counts = imports.groupby(['year', 'pair_undirected']).size().reset_index(name='import_count')

Step 1.4 Filter out the pairs with exactly 2 import records per year

clean_pairs now is a list of (year, pair_undirected), that have exactly 2 import records

In [150]:
clean_pairs = import_counts[import_counts['import_count'] == 2][['year', 'pair_undirected']]

Step 1.5 Merge 'clean_pairs' with 'import' dataframe to get 'clean_imports', containing (year, pair_undirected, usd_value, ...)

In [151]:
clean_imports = imports.merge(clean_pairs, on=['year', 'pair_undirected'])

clean_imports.head(3)

,country_a,export_or_import,country_b,year,usd_value,a_iso3,b_iso3,pair_undirected
0,Bermuda,import,Paraguay,2024.0,0.175934,BMU,PRY,"(Bermuda, Paraguay)"
1,Zimbabwe,import,Papua New Guinea,1992.0,0.000116,ZWE,PNG,"(Papua New Guinea, Zimbabwe)"
2,Zimbabwe,import,Papua New Guinea,1994.0,0.000303,ZWE,PNG,"(Papua New Guinea, Zimbabwe)"


In [152]:
# check = clean_exports[(clean_exports['pair_undirected'] == ('American Samoa', 'Uzbekistan, Republic of'))].copy()

# check.head(40)

Step 1.6: Sum the two import values for each pairing+year

We can do that because we confirmed that there are only 2 import records (1 for each direction). Both trade flows will be captured

In [153]:
scen2_summed_trade = clean_imports.groupby(['year', 'pair_undirected'], as_index=False).agg(
    total_value_usd = ('usd_value', 'sum')
)

Step 1.7: Put the countries from 'pair_undirected' back into variables country_a and country_b

In [154]:
scen2_summed_trade[['country_a', 'country_b']] = pd.DataFrame(scen2_summed_trade['pair_undirected'].tolist(), index=scen2_summed_trade.index)

scen2_summed_trade.head(3)

,year,pair_undirected,total_value_usd,country_a,country_b
0,1992.0,"(Australia, Bermuda)",0.132013,Australia,Bermuda
1,1992.0,"(Australia, Brazil)",403.192486,Australia,Brazil
2,1992.0,"(Australia, Canada)",1265.667402,Australia,Canada


Check to compare

This is how the two rows in the unsummed dataframe look like:

In [155]:
check = clean_imports[(clean_imports['pair_undirected'] == ('Bermuda', 'Paraguay')) & (clean_imports['year'] == 2024)].copy()

check.head(40)

,country_a,export_or_import,country_b,year,usd_value,a_iso3,b_iso3,pair_undirected
0,Bermuda,import,Paraguay,2024.0,0.175934,BMU,PRY,"(Bermuda, Paraguay)"
89,Paraguay,import,Bermuda,2024.0,0.000460,PRY,BMU,"(Bermuda, Paraguay)"


This is how those two rows look after summing them into one:

In [156]:
check = scen2_summed_trade[(scen2_summed_trade['pair_undirected'] == ('Bermuda', 'Paraguay')) & (scen2_summed_trade['year'] == 2024)].copy()

check.head(40)

,year,pair_undirected,total_value_usd,country_a,country_b
1522,2024.0,"(Bermuda, Paraguay)",0.176394,Bermuda,Paraguay


Step 1.8: Drop pair_undirected

In [157]:
scen2_summed_trade = scen2_summed_trade.drop('pair_undirected', axis=1)

#### Recap of Scenarion 1 and 2

In [161]:
print(f"Shape of scenario 1 dataframe: {scen1_summed_trade.shape}")
print(f"Shape of scenario 2 dataframe: {scen2_summed_trade.shape}")

Shape of scenario 1 dataframe: (277712, 4)
Shape of scenario 2 dataframe: (1559, 4)


Taking pattern distribution from the analysis to see what we have summed so far. **Bold** is what we have summed.

Pattern distribution:
  - **2 flows: exports only (both directions): 239,747 pairs (54.1%)**
  - 1 flow: A→B export only: 151,664 pairs (34.2%)
  - **3 flows: 36,830 pairs (8.3%)**
  - 2 flows: both types in A→B only: 5,909 pairs (1.3%)
  - 2 flows: A→B export + B→A import: 2,839 pairs (0.6%)
  - 1 flow: A→B import only: 2,787 pairs (0.6%)
  - 2 flows: other combination: 2,106 pairs (0.5%)
  - **all 4 flows (both directions, both types): 1,340 pairs (0.3%)**
  - **2 flows: imports only (both directions): 14 pairs (0.0%)**

#### Scenario 3: Only 1 export (1 trade flow captured), 34.2%

Step 1.1 Create a dataframe that only has export records

In [220]:
df = pd.read_csv('../data/processed/v3_imf_trade.csv')

exports = df[df['export_or_import'] == 'export'].copy()

Step 1.2 Create pair_undirected and count export records per undirected pair+year

In [221]:
exports['pair_undirected'] = exports.apply(lambda x: tuple(sorted([x['country_a'], x['country_b']])), axis=1)

export_counts = exports.groupby(['year', 'pair_undirected']).size().reset_index(name='export_count')

Step 1.4 Filter out the pairs with exactly 1 export record per year

clean_pairs now is a list of (year, pair_undirected), that have exactly 1 export record

In [222]:
clean_pairs = export_counts[export_counts['export_count'] == 1][['year', 'pair_undirected']]

Step 1.5 Merge 'clean_pairs' with 'exports' dataframe to get 'clean_exports', containing (year, pair_undirected, usd_value, ...)

In [223]:
clean_exports = exports.merge(clean_pairs, on=['year', 'pair_undirected'])

clean_exports.head(3)

,country_a,export_or_import,country_b,year,usd_value,a_iso3,b_iso3,pair_undirected
0,Holy See,export,Ukraine,2020.0,0.000228,VAT,UKR,"(Holy See, Ukraine)"
1,Holy See,export,Ukraine,2021.0,0.001300,VAT,UKR,"(Holy See, Ukraine)"
2,Holy See,export,Ukraine,2022.0,0.000881,VAT,UKR,"(Holy See, Ukraine)"


Step 3.55

In [224]:
imports = df[df['export_or_import'] == 'import'].copy()
imports['pair_undirected'] = imports.apply(lambda x: tuple(sorted([x['country_a'], x['country_b']])), axis=1)
import_pairs = set(zip(imports['year'], imports['pair_undirected']))

# Filter to keep only pairs with NO import records
clean_exports['has_import'] = clean_exports.apply(
    lambda x: (x['year'], x['pair_undirected']) in import_pairs, axis=1
)
clean_exports = clean_exports[~clean_exports['has_import']].copy()
clean_exports = clean_exports.drop('has_import', axis=1)

print(f"scen3 after removing pairs with imports: {len(clean_exports)} rows")

scen3 after removing pairs with imports: 151664 rows


Step 1.6: Rename dataset, rename 'usd_value' and 

drop: pair_undirected, usd_value, export_or_import, a_iso3, b_iso3

In [225]:
scen3_summed_trade = clean_exports.copy()

scen3_summed_trade['total_value_usd'] = scen3_summed_trade['usd_value']

scen3_summed_trade = scen3_summed_trade.drop(['usd_value', 'pair_undirected', 'export_or_import', 'a_iso3', 'b_iso3'], axis=1)

scen3_summed_trade.head(3)

,country_a,country_b,year,total_value_usd
0,Holy See,Ukraine,2020.0,0.000228
1,Holy See,Ukraine,2021.0,0.001300
2,Holy See,Ukraine,2022.0,0.000881


#### Scenario 4: Only 1 import (1 trade flow captured), 0.6%

Step 4.1 Create a dataframe that only has import records

In [212]:
df = pd.read_csv('../data/processed/v3_imf_trade.csv')

imports = df[df['export_or_import'] == 'import'].copy()

Step 4.2 Create pair_undirected and import records per undirected pair+year

In [213]:
imports['pair_undirected'] = imports.apply(lambda x: tuple(sorted([x['country_a'], x['country_b']])), axis=1)

import_counts = imports.groupby(['year', 'pair_undirected']).size().reset_index(name='import_count')

Step 4.3 Filter out the pairs with exactly 1 import record per year

clean_pairs now is a list of (year, pair_undirected), that have exactly 1 import record

In [214]:
clean_pairs = import_counts[import_counts['import_count'] == 1][['year', 'pair_undirected']]

Step 4.4 Merge 'clean_pairs' with 'exports' dataframe to get 'clean_exports', containing (year, pair_undirected, usd_value, ...)

In [215]:
clean_imports = imports.merge(clean_pairs, on=['year', 'pair_undirected'])

clean_imports.head(3)

,country_a,export_or_import,country_b,year,usd_value,a_iso3,b_iso3,pair_undirected
0,Bermuda,import,Burkina Faso,1996.0,0.000072,BMU,BFA,"(Bermuda, Burkina Faso)"
1,Bermuda,import,Burkina Faso,2009.0,0.001812,BMU,BFA,"(Bermuda, Burkina Faso)"
2,Bermuda,import,Burkina Faso,2011.0,0.044988,BMU,BFA,"(Bermuda, Burkina Faso)"


Step 4.45

In [216]:
# Remove pairs that have any export records (they don't belong in scen4)
exports = df[df['export_or_import'] == 'export'].copy()
exports['pair_undirected'] = exports.apply(lambda x: tuple(sorted([x['country_a'], x['country_b']])), axis=1)
export_pairs = set(zip(exports['year'], exports['pair_undirected']))

# Filter to keep only pairs with NO export records
clean_imports['has_export'] = clean_imports.apply(
    lambda x: (x['year'], x['pair_undirected']) in export_pairs, axis=1
)
clean_imports = clean_imports[~clean_imports['has_export']].copy()
clean_imports = clean_imports.drop('has_export', axis=1)

print(f"scen4 after removing pairs with exports: {len(clean_imports)} rows")

scen4 after removing pairs with exports: 2787 rows


Step 4.5: Rename dataset, rename 'usd_value' and 

drop pair_undirected, usd_value, export_or_import, a_iso3, b_iso3

In [217]:
scen4_summed_trade = clean_imports.copy()

scen4_summed_trade['total_value_usd'] = scen4_summed_trade['usd_value']

scen4_summed_trade = scen4_summed_trade.drop(['usd_value', 'pair_undirected', 'export_or_import', 'a_iso3', 'b_iso3'], axis=1)

scen4_summed_trade.head(3)

,country_a,country_b,year,total_value_usd
0,Bermuda,Burkina Faso,1996.0,0.000072
6,Bermuda,Greenland,1993.0,0.001773
30,Zimbabwe,Myanmar,2024.0,0.000139


#### Recap of Scenarion 3 and 4

In [218]:
print(f"Shape of scenario 3 dataframe: {scen3_summed_trade.shape}")
print(f"Shape of scenario 4 dataframe: {scen4_summed_trade.shape}")

Shape of scenario 3 dataframe: (162723, 5)
Shape of scenario 4 dataframe: (2787, 4)


Taking pattern distribution from the analysis to see what we have summed so far. **Bold** is what we have summed.

Patterns tackled by 3 and 4:
  - 2 flows: exports only (both directions): 239,747 pairs (54.1%)**
  - **1 flow: A→B export only: 151,664 pairs (34.2%)**
  - 3 flows: 36,830 pairs (8.3%)
  - 2 flows: both types in A→B only: 5,909 pairs (1.3%)
  - 2 flows: A→B export + B→A import: 2,839 pairs (0.6%)
  - **1 flow: A→B import only: 2,787 pairs (0.6%)**
  - 2 flows: other combination: 2,106 pairs (0.5%)
  - all 4 flows (both directions, both types): 1,340 pairs (0.3%)
  - 2 flows: imports only (both directions): 14 pairs (0.0%)

  Pattern distribution after 3 and 4:
  - **2 flows: exports only (both directions): 239,747 pairs (54.1%)**
  - **1 flow: A→B export only: 151,664 pairs (34.2%)**
  - **3 flows: 36,830 pairs (8.3%)**
  - 2 flows: both types in A→B only: 5,909 pairs (1.3%)
  - 2 flows: A→B export + B→A import: 2,839 pairs (0.6%)
  - **1 flow: A→B import only: 2,787 pairs (0.6%)**
  - 2 flows: other combination: 2,106 pairs (0.5%)
  - **all 4 flows (both directions, both types): 1,340 pairs (0.3%)**
  - **2 flows: imports only (both directions): 14 pairs (0.0%)**

#### Scenario 5: Import and export in one direction only (both trade flows captured), 1.3%

In [197]:
import pandas as pd

df = pd.read_csv('../data/processed/v3_imf_trade.csv')

# Create a flag for each ordered pair indicating what it has
# First, create a multi-index for fast lookup
df['direction'] = df['country_a'] + '→' + df['country_b']
df['reverse_direction'] = df['country_b'] + '→' + df['country_a']

# Pivot to see what each ordered pair has
summary = df.groupby(['year', 'country_a', 'country_b']).agg(
    has_export=('export_or_import', lambda x: 'export' in x.values),
    has_import=('export_or_import', lambda x: 'import' in x.values)
).reset_index()

# Identify ordered pairs with both export and import
both_in_one = summary[summary['has_export'] & summary['has_import']].copy()

# Create a set of all existing ordered pairs for quick membership testing
existing_pairs = set(zip(df['year'], df['country_a'], df['country_b']))

# Vectorized check for opposite direction
def check_opposite_vectorized(df_pairs):
    """Vectorized check if opposite direction exists"""
    opposites = []
    for _, row in df_pairs.iterrows():
        year, a, b = row['year'], row['country_a'], row['country_b']
        opposite_exists = (year, b, a) in existing_pairs
        opposites.append(opposite_exists)
    return opposites

# This is still a loop but over only the both_in_one pairs (45k rows, not 700k)
both_in_one['opposite_exists'] = check_opposite_vectorized(both_in_one)

# Categorize
both_in_one['category'] = 'unknown'
both_in_one.loc[both_in_one['opposite_exists'] == False, 'category'] = 'both types in one direction only'
both_in_one.loc[both_in_one['opposite_exists'] == True, 'category'] = 'has data in both directions (need further check)'

print("Breakdown:")
print(both_in_one['category'].value_counts())

# For those with opposite direction existing, further check what opposite has
opposite_exists = both_in_one[both_in_one['opposite_exists']].copy()

if len(opposite_exists) > 0:
    # Get opposite direction data in bulk
    opposite_data = []
    for _, row in opposite_exists.iterrows():
        year, a, b = row['year'], row['country_a'], row['country_b']
        opposite_data.append({'year': year, 'country_a': b, 'country_b': a})
    
    opposite_df = pd.DataFrame(opposite_data)
    
    # Merge to get opposite's info
    opposite_summary = summary.rename(columns={
        'country_a': 'temp_a', 'country_b': 'temp_b',
        'has_export': 'opp_has_export', 'has_import': 'opp_has_import'
    })
    
    opposite_exists = opposite_exists.merge(
        opposite_summary,
        left_on=['year', 'country_b', 'country_a'],
        right_on=['year', 'temp_a', 'temp_b'],
        how='left'
    )
    
    # Categorize
    opposite_exists['category'] = 'both types in A→B only'  # default
    opposite_exists.loc[opposite_exists['opp_has_export'] & ~opposite_exists['opp_has_import'], 'category'] = 'A→B both types, B→A export only'
    opposite_exists.loc[~opposite_exists['opp_has_export'] & opposite_exists['opp_has_import'], 'category'] = 'A→B both types, B→A import only'
    opposite_exists.loc[opposite_exists['opp_has_export'] & opposite_exists['opp_has_import'], 'category'] = 'all 4 flows'
    
    # Combine back
    final = pd.concat([
        both_in_one[~both_in_one['opposite_exists']],
        opposite_exists
    ], ignore_index=True)
    
    print("\nDetailed breakdown:")
    print(final['category'].value_counts())

Breakdown:
category
has data in both directions (need further check)    39510
both types in one direction only                     5909
Name: count, dtype: int64

Detailed breakdown:
category
A→B both types, B→A export only     36625
both types in one direction only     5909
all 4 flows                          2680
A→B both types, B→A import only       205
Name: count, dtype: int64


In [200]:
# Filter to the clean cases (both types in one direction only)
clean_cases = final[final['category'] == 'both types in one direction only'].copy()

# Get the actual trade rows for these cases
# First, create a key for merging
clean_cases['merge_key'] = clean_cases.apply(
    lambda x: f"{x['year']}_{x['country_a']}_{x['country_b']}", axis=1
)

# Get the original rows with both export and import for these pairs
df['merge_key'] = df.apply(
    lambda x: f"{x['year']}_{x['country_a']}_{x['country_b']}", axis=1
)

# Filter to relevant rows
relevant_rows = df[df['merge_key'].isin(clean_cases['merge_key'])].copy()

# Sum export and import for each ordered pair
scen5_summed_trade = relevant_rows.groupby(['year', 'country_a', 'country_b'], as_index=False).agg({
    'usd_value': 'sum'
})

# Rename column
scen5_summed_trade = scen5_summed_trade.rename(columns={'usd_value': 'total_value_usd'})

# Create undirected pair (sort country names)
scen5_summed_trade['country_a_sorted'] = scen5_summed_trade[['country_a', 'country_b']].min(axis=1)
scen5_summed_trade['country_b_sorted'] = scen5_summed_trade[['country_a', 'country_b']].max(axis=1)

# Keep only undirected columns
scen5_summed_trade = scen5_summed_trade[['year', 'country_a_sorted', 'country_b_sorted', 'total_value_usd']]
scen5_summed_trade = scen5_summed_trade.rename(columns={'country_a_sorted': 'country_a', 'country_b_sorted': 'country_b'})

# Drop duplicates if any (should be none)
scen5_summed_trade = scen5_summed_trade.drop_duplicates()

print(f"\nScen5 (both types in one direction only): {len(scen5_summed_trade):,} rows")
print(scen5_summed_trade.head())


Scen5 (both types in one direction only): 5,909 rows
     year       country_a  country_b  total_value_usd
0  1992.0         Algeria  Australia        20.079160
1  1992.0  American Samoa  Australia        10.438941
2  1992.0       Australia     Belize         0.106555
3  1992.0       Australia   Botswana         0.824156
4  1992.0       Australia   Cameroon         0.116373


#### Scenario 6: Export in one direction, import in other direction (one flow captured), 0.6%

In [204]:
# Extract the two patterns
pattern_export_import = classification[classification['pattern'] == '2 flows: A→B export + B→A import'].copy()
pattern_other = classification[classification['pattern'] == '2 flows: other combination'].copy()

print(f"Pattern: A→B export + B→A import: {len(pattern_export_import):,} pairs")
print(f"Pattern: other combination: {len(pattern_other):,} pairs")

# Combine both patterns
combined_patterns = pd.concat([pattern_export_import, pattern_other], ignore_index=True)

# Create a key for merging
combined_patterns['merge_key'] = combined_patterns.apply(
    lambda x: f"{x['year']}_{x['pair_undirected'][0]}_{x['pair_undirected'][1]}", axis=1
)

# Get all export records for these pairs
exports_only = df[df['export_or_import'] == 'export'].copy()
exports_only['merge_key'] = exports_only.apply(
    lambda x: f"{x['year']}_{min(x['country_a'], x['country_b'])}_{max(x['country_a'], x['country_b'])}", axis=1
)

# Merge to get export values for these pairs
merged = combined_patterns.merge(
    exports_only[['merge_key', 'usd_value']],
    on='merge_key',
    how='left'
)

# If no export, get import values
imports_only = df[df['export_or_import'] == 'import'].copy()
imports_only['merge_key'] = imports_only.apply(
    lambda x: f"{x['year']}_{min(x['country_a'], x['country_b'])}_{max(x['country_a'], x['country_b'])}", axis=1
)

# Merge imports where export is missing
merged = merged.merge(
    imports_only[['merge_key', 'usd_value']],
    on='merge_key',
    suffixes=('_export', '_import'),
    how='left'
)

# Fill with import value where export is missing
merged['total_value_usd'] = merged['usd_value_export'].fillna(merged['usd_value_import'])

# Extract country names from pair_undirected
merged['country_a'] = merged['pair_undirected'].apply(lambda x: x[0])
merged['country_b'] = merged['pair_undirected'].apply(lambda x: x[1])

Pattern: A→B export + B→A import: 2,839 pairs
Pattern: other combination: 2,106 pairs

Created dataframe with 4,945 rows


,year,country_a,country_b,total_value_usd
0,1992.0,Austria,Bermuda,5.194979
1,1992.0,Austria,Papua New Guinea,0.409902
2,1992.0,"Bahrain, Kingdom of",Bermuda,0.003000
3,1992.0,Barbados,Paraguay,0.077065
4,1992.0,Belize,Bermuda,0.016633
5,1992.0,Bermuda,Bulgaria,0.002000
6,1992.0,Bermuda,Cambodia,0.052000
7,1992.0,Bermuda,"Equatorial Guinea, Republic of",0.006000
8,1992.0,Bermuda,Greece,0.306139
9,1992.0,Bermuda,Guatemala,0.001000


In [205]:
# Create final dataframe
scen6_summed_trade = merged[['year', 'country_a', 'country_b', 'total_value_usd']].copy()

print(f"\nCreated dataframe with {len(scen6_summed_trade):,} rows")
scen6_summed_trade.head(10)


Created dataframe with 4,945 rows


,year,country_a,country_b,total_value_usd
0,1992.0,Austria,Bermuda,5.194979
1,1992.0,Austria,Papua New Guinea,0.409902
2,1992.0,"Bahrain, Kingdom of",Bermuda,0.003000
3,1992.0,Barbados,Paraguay,0.077065
4,1992.0,Belize,Bermuda,0.016633
5,1992.0,Bermuda,Bulgaria,0.002000
6,1992.0,Bermuda,Cambodia,0.052000
7,1992.0,Bermuda,"Equatorial Guinea, Republic of",0.006000
8,1992.0,Bermuda,Greece,0.306139
9,1992.0,Bermuda,Guatemala,0.001000


#### Recap of Scenario 5 and 6

In [207]:
print(f"Shape of scenario 5 dataframe: {scen5_summed_trade.shape}")
print(f"Shape of scenario 6 dataframe: {scen6_summed_trade.shape}")

Shape of scenario 5 dataframe: (5909, 4)
Shape of scenario 6 dataframe: (4945, 4)


Taking pattern distribution from the analysis to see what we have summed so far. **Bold** is what we have handled.

Patterns tackled by 5 and 6:
  - 2 flows: exports only (both directions): 239,747 pairs (54.1%)**
  - 1 flow: A→B export only: 151,664 pairs (34.2%)
  - 3 flows: 36,830 pairs (8.3%)
  - **2 flows: both types in A→B only: 5,909 pairs (1.3%)**
  - **2 flows: A→B export + B→A import: 2,839 pairs (0.6%)**
  - 1 flow: A→B import only: 2,787 pairs (0.6%)
  - **2 flows: other combination: 2,106 pairs (0.5%)**
  - all 4 flows (both directions, both types): 1,340 pairs (0.3%)
  - 2 flows: imports only (both directions): 14 pairs (0.0%)

  Pattern distribution after 5 and 6:
  - **2 flows: exports only (both directions): 239,747 pairs (54.1%)**
  - **1 flow: A→B export only: 151,664 pairs (34.2%)**
  - **3 flows: 36,830 pairs (8.3%)**
  - **2 flows: both types in A→B only: 5,909 pairs (1.3%)**
  - **2 flows: A→B export + B→A import: 2,839 pairs (0.6%)**
  - **1 flow: A→B import only: 2,787 pairs (0.6%)**
  - **2 flows: other combination: 2,106 pairs (0.5%)**
  - **all 4 flows (both directions, both types): 1,340 pairs (0.3%)**
  - **2 flows: imports only (both directions): 14 pairs (0.0%)**

### Scenario's summarized

| Scenario | Pattern | Trade flows captured | Method |
|----------|---------|---------------------|--------|
| scen1 | Exports only (both directions) | 2 flows (export A→B + export B→A) | Sum both exports |
| scen2 | Imports only (both directions) | 2 flows (import A→B + import B→A) | Sum both imports |
| scen3 | Export only (one direction) | 1 flow (export A→B only) | Single export value |
| scen4 | Import only (one direction) | 1 flow (import A→B only) | Single import value |
| scen5 | Both types in one direction only | 2 flows (export + import in same direction) | Sum export + import |
| scen6 | Export one way + import the other | 1 unique flow (duplicate records) | Single value (export or import) |

#### Merging the 6 scenario's

In [235]:
# List all your scenario dataframes
scenarios = [scen1_summed_trade, scen2_summed_trade, scen3_summed_trade, scen4_summed_trade, scen5_summed_trade, scen6_summed_trade]

# Add priority column to each scenario (lower number = higher priority)
# Priority based on reliability: exporter-reported data first
priority_order = {
    'scen1': 1,  # Exports both sides (most reliable)
    'scen5': 2,  # Both types same direction
    'scen3': 3,  # Export one direction
    'scen6': 4,  # Export + import opposite
    'scen4': 5,  # Import only one direction
    'scen2': 6   # Imports both sides (least reliable)
}

# Assign priority to each scenario dataframe
scen1_summed_trade['priority'] = priority_order['scen1']
scen2_summed_trade['priority'] = priority_order['scen2']
scen3_summed_trade['priority'] = priority_order['scen3']
scen4_summed_trade['priority'] = priority_order['scen4']
scen5_summed_trade['priority'] = priority_order['scen5']
scen6_summed_trade['priority'] = priority_order['scen6']

# Combine all into one dataframe
all_scenarios = pd.concat(scenarios, ignore_index=True)

print("=" * 60)
print("MERGING ALL SCENARIO DATAFRAMES")
print("=" * 60)

print(f"\nTotal rows before deduplication: {len(all_scenarios):,}")

# Create pair_key for identifying same year and country pair
all_scenarios['pair_key'] = all_scenarios.apply(
    lambda x: f"{x['year']}_{min(x['country_a'], x['country_b'])}_{max(x['country_a'], x['country_b'])}", axis=1
)

# Sort by priority first (lower number = higher priority), then by value
all_scenarios_sorted = all_scenarios.sort_values(['priority', 'total_value_usd'])

# Remove duplicates: keep first occurrence based on priority order
scen_final = all_scenarios_sorted.drop_duplicates(subset=['pair_key'], keep='first')

# Calculate statistics
total_duplicates = len(all_scenarios) - len(scen_final)
duplicate_pairs = all_scenarios.duplicated(subset=['pair_key'], keep=False)
unique_duplicate_pairs = all_scenarios[duplicate_pairs]['pair_key'].nunique()

print(f"\nRows after priority-based deduplication: {len(scen_final):,}")
print(f"Rows removed (lower priority duplicates): {total_duplicates:,}")
print(f"Unique country pairs with duplicates: {unique_duplicate_pairs:,}")

# Show which scenarios contributed to the final dataframe
print("\n" + "=" * 60)
print("SOURCE OF FINAL VALUES (by priority)")
print("=" * 60)
source_counts = scen_final['priority'].map({1:'scen1', 2:'scen5', 3:'scen3', 4:'scen6', 5:'scen4', 6:'scen2'}).value_counts()
for source, count in source_counts.items():
    pct = (count / len(scen_final)) * 100
    print(f"  {source}: {count:,} rows ({pct:.1f}%)")

# Show examples of what was resolved
print("\n" + "=" * 60)
print("EXAMPLES OF RESOLVED DUPLICATES (lower priority values discarded)")
print("=" * 60)

# Find examples where multiple scenarios had different values
duplicate_examples = all_scenarios[all_scenarios.duplicated(subset=['pair_key'], keep=False)]
shown = 0
for pair_key, group in duplicate_examples.groupby('pair_key'):
    if len(group['total_value_usd'].unique()) > 1 and shown < 5:
        year, a, b = pair_key.split('_')
        print(f"\nPair: {year}, {a} - {b}")
        print(f"  Kept (priority {group['priority'].min()}): {group[group['priority'] == group['priority'].min()]['total_value_usd'].iloc[0]}")
        print(f"  Discarded values:")
        for _, row in group[group['priority'] > group['priority'].min()].iterrows():
            print(f"    - Priority {row['priority']} ({['scen1','scen5','scen3','scen6','scen4','scen2'][int(row['priority'])-1]}): {row['total_value_usd']}")
        shown += 1

if shown == 0:
    print("\nNo conflicting duplicates found!")

# Drop temporary columns
scen_final = scen_final.drop(['pair_key', 'priority'], axis=1)

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Final dataframe shape: {scen_final.shape}")
print(f"\n' scen_final ' is ready for trade dependency calculations")

MERGING ALL SCENARIO DATAFRAMES

Total rows before deduplication: 444,576

Rows after priority-based deduplication: 443,236
Rows removed (lower priority duplicates): 1,340
Unique country pairs with duplicates: 1,340

SOURCE OF FINAL VALUES (by priority)
  scen1: 277,712 rows (62.7%)
  scen3: 151,664 rows (34.2%)
  scen5: 5,909 rows (1.3%)
  scen6: 4,945 rows (1.1%)
  scen4: 2,787 rows (0.6%)
  scen2: 219 rows (0.0%)

EXAMPLES OF RESOLVED DUPLICATES (lower priority values discarded)

Pair: 1992.0, Australia - Brazil
  Kept (priority 1): 381.29788986936
  Discarded values:
    - Priority 6 (scen2): 403.192486022503

Pair: 1992.0, Australia - Canada
  Kept (priority 1): 1261.785583469845
  Discarded values:
    - Priority 6 (scen2): 1265.667402245433

Pair: 1992.0, Australia - Czechoslovakia
  Kept (priority 1): 43.8309149488192
  Discarded values:
    - Priority 6 (scen2): 63.20985424956859

Pair: 1992.0, Australia - Dominican Republic
  Kept (priority 1): 0.36663546035385797
  Discarded

## Phase 1 is done

We have the dataset where each undirected pair has one total_value_usd capturing two flows when available, and otherwise capturing one flow.

In [242]:
print(scen_final.shape)
scen_final.head(3)

(443236, 6)


,year,total_value_usd,country_a,country_b,a_iso3,b_iso3
0,2012.0,0.000024,Barbados,"South Sudan, Republic of",BRB,SSD
1,2021.0,0.000025,Côte d'Ivoire,"Micronesia, Federated States of",CIV,FSM
2,2013.0,0.000035,Denmark,Tuvalu,DNK,TUV


Add back iso3 codes using df

In [240]:
# Create country to iso3 mapping
country_iso = pd.concat([
    df[['country_a', 'a_iso3']].rename(columns={'country_a': 'country', 'a_iso3': 'iso3'}),
    df[['country_b', 'b_iso3']].rename(columns={'country_b': 'country', 'b_iso3': 'iso3'})
]).drop_duplicates('country')

# Add iso3 columns
scen_final = scen_final.merge(country_iso, left_on='country_a', right_on='country', how='left').drop('country', axis=1)
scen_final = scen_final.merge(country_iso, left_on='country_b', right_on='country', how='left').drop('country', axis=1)
scen_final = scen_final.rename(columns={'iso3_x': 'a_iso3', 'iso3_y': 'b_iso3'})

scen_final.head()

,year,total_value_usd,country_a,country_b,a_iso3,b_iso3
0,2012.0,0.000024,Barbados,"South Sudan, Republic of",BRB,SSD
1,2021.0,0.000025,Côte d'Ivoire,"Micronesia, Federated States of",CIV,FSM
2,2013.0,0.000035,Denmark,Tuvalu,DNK,TUV
3,2016.0,0.000055,"Latvia, Republic of","São Tomé and Príncipe, Democratic Republic of",LVA,STP
4,2016.0,0.000060,Gabon,Zambia,GAB,ZMB


Save this dataset to easily use in phase 2

In [243]:
scen_final.to_csv('../data/processed/trade_dependency_engineering/p1_trade_engineering.csv', index=False)

## Phase 2: Dividing by GDP

In [413]:
trade = pd.read_csv('../data/processed/trade_dependency_engineering/p1_trade_engineering.csv')

In [414]:
trade.head()

,year,total_value_usd,country_a,country_b,a_iso3,b_iso3
0,2012.0,0.000024,Barbados,"South Sudan, Republic of",BRB,SSD
1,2021.0,0.000025,Côte d'Ivoire,"Micronesia, Federated States of",CIV,FSM
2,2013.0,0.000035,Denmark,Tuvalu,DNK,TUV
3,2016.0,0.000055,"Latvia, Republic of","São Tomé and Príncipe, Democratic Republic of",LVA,STP
4,2016.0,0.000060,Gabon,Zambia,GAB,ZMB


In [415]:
controls = pd.read_csv('../data/processed/controls/controls_merged.csv')

In [416]:
controls.head()

,iso3,year,gdp_per_capita,gdp_per_capita_log,population,population_log,v2x_polyarchy,armed_conflict,conflict_intensity
0,ABW,1992,13892.605143,9.539112,69005.0,11.141934,NaN,0,0
1,ABW,1993,14700.959808,9.595668,73685.0,11.207555,NaN,0,0
2,ABW,1994,16055.287787,9.683794,77595.0,11.259258,NaN,0,0
3,ABW,1995,16548.717387,9.714064,79805.0,11.287341,NaN,0,0
4,ABW,1996,16620.954556,9.718420,83021.0,11.326849,NaN,0,0


Step 1: Merge GDP column from controls into the trade df

In [417]:
gdp = controls[['iso3', 'year', 'gdp_per_capita']].copy()

In [418]:
# Merge GDP for country_a
trade = trade.merge(
    gdp,
    left_on=['a_iso3', 'year'],
    right_on=['iso3', 'year'],
    how='left'
).rename(columns={'gdp_per_capita': 'a_gdp'}).drop('iso3', axis=1)

# Merge GDP for country_b
trade = trade.merge(
    gdp,
    left_on=['b_iso3', 'year'],
    right_on=['iso3', 'year'],
    how='left'
).rename(columns={'gdp_per_capita': 'b_gdp'}).drop('iso3', axis=1)

# Check for missing values
print(f"Missing a_gdp: {trade['a_gdp'].isna().sum()}")
print(f"Missing b_gdp: {trade['b_gdp'].isna().sum()}")

# Reorder columns
trade = trade[['year', 'country_a', 'a_iso3', 'a_gdp', 'country_b', 'b_iso3', 'b_gdp', 'total_value_usd']]

trade.head()

Missing a_gdp: 11946
Missing b_gdp: 13663


,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060


Step 2: Check missingness in GDP

In [254]:
# Check which countries have missing GDP
missing_a = trade[trade['a_gdp'].isna()][['country_a', 'a_iso3', 'year']].drop_duplicates()
missing_b = trade[trade['b_gdp'].isna()][['country_b', 'b_iso3', 'year']].drop_duplicates()

print("Countries with missing GDP as country_a:")
print(missing_a.head(10))
print(f"\nTotal unique countries missing as a: {missing_a['country_a'].nunique()}")

print("\nCountries with missing GDP as country_b:")
print(missing_b.head(10))
print(f"\nTotal unique countries missing as b: {missing_b['country_b'].nunique()}")

# Check if ISO3 codes exist in GDP data
gdp_iso3s = set(gdp['iso3'].unique())
trade_iso3s_a = set(trade['a_iso3'].dropna().unique())
trade_iso3s_b = set(trade['b_iso3'].dropna().unique())

print(f"\nISO3 codes in GDP: {len(gdp_iso3s)}")
print(f"ISO3 codes in trade (a_iso3): {len(trade_iso3s_a)}")
print(f"ISO3 codes in trade (b_iso3): {len(trade_iso3s_b)}")

# Find ISO3 codes in trade that are NOT in GDP
missing_iso_a = trade_iso3s_a - gdp_iso3s
missing_iso_b = trade_iso3s_b - gdp_iso3s

print(f"\nISO3 codes missing from GDP (a_iso3): {missing_iso_a}")
print(f"ISO3 codes missing from GDP (b_iso3): {missing_iso_b}")

Countries with missing GDP as country_a:
                                            country_a a_iso3    year
5                         Falkland Islands (Malvinas)    FLK  2021.0
8   Anguilla, United Kingdom-British Overseas Terr...    AIA  2023.0
10                                           Holy See    VAT  2001.0
19                              Eritrea, The State of    ERI  2024.0
24             Korea, Democratic People's Republic of    PRK  2021.0
44                        Falkland Islands (Malvinas)    FLK  2023.0
52                                          Gibraltar    GIB  2007.0
57  Montserrat, United Kingdom-British Overseas Te...    MSR  2021.0
61  Anguilla, United Kingdom-British Overseas Terr...    AIA  2014.0
68  Anguilla, United Kingdom-British Overseas Terr...    AIA  2021.0

Total unique countries missing as a: 27

Countries with missing GDP as country_b:
                                  country_b b_iso3    year
21                                 Holy See    VAT  2022.0

The analysis shows that 4 countries/territories have no data in the GDP dataset. 

For the rest (max 29 countries), there are only certain years missing the GDP.

We can drop these rows containing missing values in the GDP, since just a trade value does not tell us enough about its dependency.

Step 3: Drop rows that miss GDP

In [257]:
trade_clean = trade.dropna(subset=['a_gdp', 'b_gdp']).copy()

print(f"Original rows: {len(trade):,}")
print(f"Clean rows: {len(trade_clean):,}")
print(f"Rows removed: {len(trade) - len(trade_clean):,}")

Original rows: 443,236
Clean rows: 417,841
Rows removed: 25,395


,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,dependency_a_pct,dependency_b_pct
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06


In [263]:
# Calculate trade dependency percentages
trade_clean['a_dependency_pct'] = (trade_clean['total_value_usd'] / trade_clean['a_gdp']) * 100
trade_clean['b_dependency_pct'] = (trade_clean['total_value_usd'] / trade_clean['b_gdp']) * 100

trade_clean.head()



,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,a_dependency_pct,b_dependency_pct
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06


In [260]:
check = trade_clean[(trade_clean['a_iso3'] == 'NLD') & (trade_clean['year'] == 2024)]

check.head(5)

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,dependency_a_pct,dependency_b_pct
71149,2024.0,"Netherlands, The",NLD,67520.421896,"São Tomé and Príncipe, Democratic Republic of",STP,3490.568835,2.726443,0.004038,0.078109
86775,2024.0,"Netherlands, The",NLD,67520.421896,St. Kitts and Nevis,KNA,23960.653436,4.750317,0.007035,0.019825
90351,2024.0,"Netherlands, The",NLD,67520.421896,"Timor-Leste, Democratic Republic of",TLS,1331.970513,5.381292,0.007970,0.404010
110196,2024.0,"Netherlands, The",NLD,67520.421896,St. Vincent and the Grenadines,VCT,11501.226519,10.119669,0.014988,0.087988
113218,2024.0,"Netherlands, The",NLD,67520.421896,St. Lucia,LCA,14181.630335,11.073350,0.016400,0.078082


## Phase 2 is done

In [265]:
trade_clean.shape

(417841, 10)

Save it

In [264]:
trade_clean.to_csv('../data/processed/trade_dependency_engineering/p2_trade_engineering.csv', index=False)

## Phase 3: Income classification

In [316]:
import pandas as pd

trade = pd.read_csv('../data/processed/trade_dependency_engineering/p2_trade_engineering.csv')

ic = pd.read_csv('../data/processed/wb_income_classification.csv')

In [317]:
print(trade.shape)
trade.head()

(417841, 10)


,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,a_dependency_pct,b_dependency_pct
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06


In [318]:
print(ic.shape)
ic.head()

(7877, 4)


,country_name,year,classification,country_iso3
0,Afghanistan,1987,L,AFG
1,Afghanistan,1988,L,AFG
2,Afghanistan,1989,L,AFG
3,Afghanistan,1990,L,AFG
4,Afghanistan,1991,L,AFG


Step 1: in ic, leave out country_name and drop duplicates if there are any 

In [319]:
ic = ic[['country_iso3', 'year', 'classification']].drop_duplicates()
print(ic.shape)

(7861, 3)


Step 2: Merge the two dataframes

In [320]:
trade = trade.merge(
    ic,
    left_on=['a_iso3', 'year'],
    right_on=['country_iso3', 'year'],
    how='left'
).drop('country_iso3', axis=1).rename(columns={'classification': 'a_ic'})

trade = trade.merge(
    ic,
    left_on=['b_iso3', 'year'],
    right_on=['country_iso3', 'year'],
    how='left'
).drop('country_iso3', axis=1).rename(columns={'classification': 'b_ic'})

In [321]:
trade.head()

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,a_dependency_pct,b_dependency_pct,a_ic,b_ic
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06,H,L
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07,LM,LM
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07,H,UM
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06,H,LM
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06,UM,LM


Step 3: Check and handle missingness in income classification

In [322]:
print(f"Missing a_ic: {trade['a_ic'].isna().sum()}")
print(f"Missing b_ic: {trade['b_ic'].isna().sum()}")

# Rows where either a_ic or b_ic is missing
missing_either = trade['a_ic'].isna() | trade['b_ic'].isna()
print(f"Rows with missing in either a_ic or b_ic: {missing_either.sum()}")

# Percentage of total rows affected
total_rows = len(trade)
print(f"\nPercentage of rows with any missing classification: {missing_either.sum() / total_rows * 100:.2f}%")

Missing a_ic: 991
Missing b_ic: 1692
Rows with missing in either a_ic or b_ic: 2681

Percentage of rows with any missing classification: 0.64%


In [323]:
# Count unique countries missing a_ic
missing_a_countries = trade[trade['a_ic'].isna()]['country_a'].nunique()
print(f"Unique countries missing a_ic: {missing_a_countries}")

# Show which countries
missing_a_list = trade[trade['a_ic'].isna()]['country_a'].unique()
print(f"Countries missing a_ic: {missing_a_list}")

# Count unique countries missing b_ic
missing_b_countries = trade[trade['b_ic'].isna()]['country_b'].nunique()
print(f"\nUnique countries missing b_ic: {missing_b_countries}")

# Show which countries
missing_b_list = trade[trade['b_ic'].isna()]['country_b'].unique()
print(f"Countries missing b_ic: {missing_b_list}")

# Check if these missing countries exist in your income data at all
all_iso3_in_income = set(ic['country_iso3'].unique())
missing_a_iso3 = set(trade[trade['a_ic'].isna()]['a_iso3'].unique())
missing_b_iso3 = set(trade[trade['b_ic'].isna()]['b_iso3'].unique())

print(f"\nMissing a_iso3 that don't exist in income data: {missing_a_iso3 - all_iso3_in_income}")
print(f"Missing b_iso3 that don't exist in income data: {missing_b_iso3 - all_iso3_in_income}")

Unique countries missing a_ic: 5
Countries missing a_ic: ['Tuvalu' 'Nauru, Republic of'
 'Ethiopia, The Federal Democratic Republic of'
 'Curaçao, Kingdom of the Netherlands'
 'Venezuela, República Bolivariana de']

Unique countries missing b_ic: 6
Countries missing b_ic: ['Nauru, Republic of' 'Venezuela, República Bolivariana de' 'Tuvalu'
 'Ethiopia, The Federal Democratic Republic of' 'Montenegro'
 'Palau, Republic of']

Missing a_iso3 that don't exist in income data: set()
Missing b_iso3 that don't exist in income data: set()


All countries in the trade data exist in the income data, that is good. But there is missingness in some years.

In [324]:
# Check which years are missing for these countries
print("Missing years for a_ic:")
for country in missing_a_list:
    years_missing = trade[(trade['country_a'] == country) & (trade['a_ic'].isna())]['year'].unique()
    print(f"  {country}: {sorted(years_missing)}")

print("\nMissing years for b_ic:")
for country in missing_b_list:
    years_missing = trade[(trade['country_b'] == country) & (trade['b_ic'].isna())]['year'].unique()
    print(f"  {country}: {sorted(years_missing)}")

Missing years for a_ic:
  Tuvalu: [np.float64(2000.0), np.float64(2001.0), np.float64(2002.0), np.float64(2003.0), np.float64(2004.0), np.float64(2005.0), np.float64(2006.0), np.float64(2007.0), np.float64(2008.0)]
  Nauru, Republic of: [np.float64(1992.0), np.float64(2000.0), np.float64(2001.0), np.float64(2002.0), np.float64(2003.0), np.float64(2004.0), np.float64(2005.0), np.float64(2006.0), np.float64(2007.0), np.float64(2008.0), np.float64(2009.0), np.float64(2010.0), np.float64(2011.0), np.float64(2012.0), np.float64(2013.0), np.float64(2014.0)]
  Ethiopia, The Federal Democratic Republic of: [np.float64(2024.0)]
  Curaçao, Kingdom of the Netherlands: [np.float64(2009.0)]
  Venezuela, República Bolivariana de: [np.float64(2020.0), np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0)]

Missing years for b_ic:
  Nauru, Republic of: [np.float64(1992.0), np.float64(1993.0), np.float64(1994.0), np.float64(1995.0), np.float64(1996.0), np.float64(1997.0), np.fl

##### My modeling choice

To deal with the missingess we could potentially forward fill, which means imputing the missing values with the classification thats closest to it in year. It is possible since income classifications would not wildly change each year, and these countries don't seem to have gone through crazy changes within their missing years. 

But I'm still uncomfortable with doing so and potentially making wrong assumptions. I'll sacrifice 0.64% of the data to have a possibly worse performing model but atleast it won't be well performing on wrong assumptions

In [325]:
# Drop rows where either a_ic or b_ic is missing
trade_clean = trade.dropna(subset=['a_ic', 'b_ic']).copy()

print(f"Original rows: {len(trade):,}")
print(f"Rows after dropping: {len(trade_clean):,}")
print(f"Rows removed: {len(trade) - len(trade_clean):,}")
print(f"Percentage removed: {(len(trade) - len(trade_clean)) / len(trade) * 100:.2f}%")

Original rows: 418,371
Rows after dropping: 415,690
Rows removed: 2,681
Percentage removed: 0.64%


Step 4: Assign integer numbers to each classification

In [326]:
# Define your mapping
classification_mapping = {
    'L': '1',
    'LM': '2',
    'UM': '3',
    'H': '4'
}

# Apply mapping to both columns
trade_clean['a_ic'] = trade_clean['a_ic'].map(classification_mapping)
trade_clean['b_ic'] = trade_clean['b_ic'].map(classification_mapping)

print("\nCombined unique countries per classification:")
all_countries_a = trade_clean[['a_ic', 'country_a']].rename(columns={'a_ic': 'ic', 'country_a': 'country'})
all_countries_b = trade_clean[['b_ic', 'country_b']].rename(columns={'b_ic': 'ic', 'country_b': 'country'})
all_countries = pd.concat([all_countries_a, all_countries_b]).drop_duplicates()
combined_counts = all_countries.groupby('ic')['country'].nunique()
print(combined_counts)


Combined unique countries per classification:
ic
1     76
2    120
3    101
4     79
Name: country, dtype: int64


In [327]:
trade_clean.head(3)

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,a_dependency_pct,b_dependency_pct,a_ic,b_ic
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06,4,1
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07,2,2
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07,4,3


Step 5: Check

In [328]:
check = trade_clean[(trade_clean['a_iso3'] == 'NLD') & (trade_clean['year'] == 2024)]

check.head(5)

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,a_dependency_pct,b_dependency_pct,a_ic,b_ic
66210,2024.0,"Netherlands, The",NLD,67520.421896,"São Tomé and Príncipe, Democratic Republic of",STP,3490.568835,2.726443,0.004038,0.078109,4,2
81206,2024.0,"Netherlands, The",NLD,67520.421896,St. Kitts and Nevis,KNA,23960.653436,4.750317,0.007035,0.019825,4,4
84647,2024.0,"Netherlands, The",NLD,67520.421896,"Timor-Leste, Democratic Republic of",TLS,1331.970513,5.381292,0.007970,0.404010,4,2
103877,2024.0,"Netherlands, The",NLD,67520.421896,St. Vincent and the Grenadines,VCT,11501.226519,10.119669,0.014988,0.087988,4,3
106818,2024.0,"Netherlands, The",NLD,67520.421896,St. Lucia,LCA,14181.630335,11.073350,0.016400,0.078082,4,3


In [329]:
trade_clean.to_csv('../data/processed/trade_dependency_engineering/p3_trade_engineering.csv', index=False)

### Phase 4: Trade share of trade with all partners

Now we will calculate the share that that dyadic trade relation is of the total trade of a certain country with all their partners that year. 

We will change the trade value column from 'total_value_usd' to 'total_trade_relation_value' as to prevent confusion. This means that total_trade_relation_value captures the two trade flows summed for that year, for that specific pairing.

We will add **4** new columns in total. 

The first two will be the total (summed) trade that country a (or country b) has with all their partners that year. 

The other two columns will be calculated like this:

- a_trade_share = total_trade_relation_value / a_total_trade_all_partners
- b_trade_share = total_trade_relation_value / b_total_trade_all_partners

**Example:**
- country_a = china
- country_b = us
- year = 2020
- total_trade_relation_value = 2000
- a_total_trade_all_partners = 400000
- b_total_trade_all_partners = 600000
- a_trade_share = 0.005
- b_trade_share = 0.003

In [419]:
trade = pd.read_csv('../data/processed/trade_dependency_engineering/p3_trade_engineering.csv')

trade.head()

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_value_usd,a_dependency_pct,b_dependency_pct,a_ic,b_ic
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06,4,1
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07,2,2
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07,4,3
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06,4,2
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06,3,2


In [420]:
# rename total_value_usd to total_trade_relation_value for clarity
trade = trade.rename(columns={'total_value_usd': 'total_trade_relation_value'})

# Calculate total trade for each country per year (both as exporter and importer)
# Step 1: Get trade where country is a_iso3
total_a = trade.groupby(['year', 'a_iso3'])['total_trade_relation_value'].sum().reset_index()
total_a.columns = ['year', 'iso3', 'trade_volume']

# Step 2: Get trade where country is b_iso3
total_b = trade.groupby(['year', 'b_iso3'])['total_trade_relation_value'].sum().reset_index()
total_b.columns = ['year', 'iso3', 'trade_volume']

# Step 3: Combine and sum both sides
combined = pd.concat([total_a, total_b], ignore_index=True)
country_total = combined.groupby(['year', 'iso3'])['trade_volume'].sum().reset_index()
country_total.columns = ['year', 'iso3', 'country_total_trade']

# Step 4: Add total trade for country_a
trade = trade.merge(country_total, left_on=['year', 'a_iso3'], right_on=['year', 'iso3'], how='left')
trade = trade.drop('iso3', axis=1)
trade = trade.rename(columns={'country_total_trade': 'a_total_trade_all_partners'})

# Step 5: Add total trade for country_b
trade = trade.merge(country_total, left_on=['year', 'b_iso3'], right_on=['year', 'iso3'], how='left')
trade = trade.drop('iso3', axis=1)
trade = trade.rename(columns={'country_total_trade': 'b_total_trade_all_partners'})

# Calculate trade shares
trade['a_trade_share'] = trade['total_trade_relation_value'] / trade['a_total_trade_all_partners']
trade['b_trade_share'] = trade['total_trade_relation_value'] / trade['b_total_trade_all_partners']

In [421]:
trade.head()

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_trade_relation_value,a_dependency_pct,b_dependency_pct,a_ic,b_ic,a_total_trade_all_partners,b_total_trade_all_partners,a_trade_share,b_trade_share
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06,4,1,1825.094587,232.631092,1.315000e-08,1.031676e-07
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07,2,2,43604.470815,513.942559,5.733357e-10,4.864357e-08
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07,4,3,205206.269437,108.816958,1.705601e-10,3.216410e-07
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06,4,2,32298.999051,160.416332,1.702839e-09,3.428579e-07
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06,3,2,12070.608990,11552.920906,4.970752e-09,5.193492e-09


### Phase 4 is done

In [423]:
trade.to_csv('../data/processed/trade_dependency_engineering/p4_trade_engineering.csv', index=False)

Here is a function to easily check what the top 10 trading partners are for a country and year.

In [411]:
def country_trade_summary(country_name, year):
    """
    Get trade summary for a specific country and year.
    
    Parameters:
    country_name: str (e.g., 'United Kingdom', 'United States', 'Germany')
    year: float or int (e.g., 2024, 2020)
    """
    
    # Get all trade for the country in the specified year (both as a and b)
    country_trade = trade[
        ((trade['country_a'] == country_name) | (trade['country_b'] == country_name)) & 
        (trade['year'] == year)
    ].copy()
    
    if len(country_trade) == 0:
        print(f"No data found for {country_name} in {year}")
        return None
    
    # Get partner name and trade value
    country_trade['partner'] = country_trade.apply(
        lambda x: x['country_b'] if x['country_a'] == country_name else x['country_a'], axis=1
    )
    country_trade['trade_value'] = country_trade['total_trade_relation_value']
    
    # Get country's total trade
    if country_name in country_trade['country_a'].values:
        country_total = country_trade[country_trade['country_a'] == country_name]['a_total_trade_all_partners'].iloc[0]
    else:
        country_total = country_trade[country_trade['country_b'] == country_name]['b_total_trade_all_partners'].iloc[0]
    
    # Sort by trade value descending
    top_partners = country_trade[['partner', 'trade_value']].sort_values('trade_value', ascending=False).head(10)
    top_partners['trade_share'] = (top_partners['trade_value'] / country_total) * 100
    
    print(f"{country_name} total trade with all partners in {year}: ${country_total:,.0f} million (${country_total/1000:.1f} billion)")
    print(f"\nTop 10 trading partners:")
    
    return top_partners

Calling that function. 

trade_value shows what the summed trade flows are between target country and partner for target year

trade_share is the percentage of target country's trade with that partner

So for example when calling for United States in 2024:

- partner = Mexico
- trade_value = 846751.877045
- trade_share = 16.302859

16% of US trade was with Mexico in 2024

In [422]:
country_trade_summary('United States', 2024)

United States total trade with all partners in 2024: $5,193,886 million ($5193.9 billion)

Top 10 trading partners:


,partner,trade_value,trade_share
268234,Mexico,846751.877045,16.302859
268229,Canada,784512.157927,15.104533
268221,"China, People's Republic of",668770.972341,12.876120
268108,Germany,250143.401327,4.816113
268078,Japan,221235.142125,4.259530
268026,"Korea, Republic of",193953.398418,3.734264
267950,United Kingdom,153629.175912,2.957885
267889,"Netherlands, The",136522.323077,2.628520
267868,Vietnam,132472.342786,2.550544
267825,India,122667.167968,2.361761


## Phase 5: according to the orginal calculation plan

In [7]:
import pandas as pd
import numpy as np
trade = pd.read_csv('../data/processed/trade_dependency_engineering/p4_trade_engineering.csv')

In [10]:
trade.head()   

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_trade_relation_value,a_dependency_pct,b_dependency_pct,a_ic,b_ic,a_total_trade_all_partners,b_total_trade_all_partners,a_trade_share,b_trade_share,country_a_multiplier,country_b_multiplier
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06,4,1,1825.094587,232.631092,1.315000e-08,1.031676e-07,0,3
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07,2,2,43604.470815,513.942559,5.733357e-10,4.864357e-08,0,0
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07,4,3,205206.269437,108.816958,1.705601e-10,3.216410e-07,0,1
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06,4,2,32298.999051,160.416332,1.702839e-09,3.428579e-07,0,2
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06,3,2,12070.608990,11552.920906,4.970752e-09,5.193492e-09,0,1


Calculate difference in income class, create multiplier columns

In [9]:
trade['country_a_multiplier'] = np.where(
    trade['a_ic'] < trade['b_ic'],
    trade['b_ic'] - trade['a_ic'],
    0
)

trade['country_b_multiplier'] = np.where(
    trade['b_ic'] < trade['a_ic'],
    trade['a_ic'] - trade['b_ic'],
    0
)

Rename the current dependency percentage we have for clarity

In [14]:
trade = trade.rename(columns={
    'a_dependency_pct': 'a_dependency_unweighted',
    'b_dependency_pct': 'b_dependency_unweighted'
})

Create the weighted columns

In [16]:
trade['a_dependency_weighted'] = trade['a_dependency_unweighted'] * trade['country_a_multiplier']
trade['b_dependency_weighted'] = trade['b_dependency_unweighted'] * trade['country_b_multiplier']

In [17]:
trade.head()

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_trade_relation_value,a_dependency_unweighted,b_dependency_unweighted,a_ic,b_ic,a_total_trade_all_partners,b_total_trade_all_partners,a_trade_share,b_trade_share,country_a_multiplier,country_b_multiplier,a_dependency_weighted,b_dependency_weighted
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06,4,1,1825.094587,232.631092,1.315000e-08,1.031676e-07,0,3,0.0,6.490811e-06
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07,2,2,43604.470815,513.942559,5.733357e-10,4.864357e-08,0,0,0.0,0.000000e+00
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07,4,3,205206.269437,108.816958,1.705601e-10,3.216410e-07,0,1,0.0,9.970895e-07
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06,4,2,32298.999051,160.416332,1.702839e-09,3.428579e-07,0,2,0.0,7.666199e-06
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06,3,2,12070.608990,11552.920906,4.970752e-09,5.193492e-09,0,1,0.0,4.842282e-06


Save

In [18]:
trade.to_csv('../data/processed/trade_dependency_engineering/p5_trade_engineering.csv', index=False)